# Feature engineering — internações respiratórias x qualidade do ar

Este notebook constrói o dataset final de modelagem para prever o número de internações por doenças respiratórias em `D0`, combinando informações da própria série de internações com variáveis atmosféricas e componentes sazonais.

A construção das features segue a lógica definida em `featureDefinition.md` e respeita o princípio de **anti-leakage**: nenhuma variável derivada da série de internações utiliza informação do próprio `D0` ou de datas futuras.

**Entradas**
- `respiratory_hospitalization_time_series.parquet`
- `serie_diaria_qualidade_ar_rio_de_janeiro.parquet`

**Saída**
- `Data/GoldData/modelDataset.parquet`


## Importações e configuração inicial

Nesta etapa são importadas as bibliotecas necessárias e definidos os parâmetros básicos de execução do notebook.


In [8]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
from statsmodels.tsa.seasonal import STL

warnings.filterwarnings('ignore')

## Caminhos dos dados e carregamento das bases

Aqui são definidos os caminhos das duas bases utilizadas no processo e realizado o carregamento inicial dos dados em memória.


In [9]:
BASE_DIR = Path('../../Data')

PATH_INTERNACOES = 'https://raw.githubusercontent.com/AILAB-CEFET-RJ/qualiar/refs/heads/Refactoring-And-Documentation/Data/IntermediaryData/DataSus/respiratory_hospitalization_time_series.parquet'
PATH_QUALIAR = 'https://raw.githubusercontent.com/AILAB-CEFET-RJ/qualiar/refs/heads/Refactoring-And-Documentation/Data/IntermediaryData/MonitorAr/DailyQualiarRj/serie_diaria_qualidade_ar_rio_de_janeiro.parquet'

PATH_OUTPUT = BASE_DIR / 'GoldData' / 'modelDataset.parquet'

# Carrega as bases de dados
df_internacoes = pd.read_parquet(PATH_INTERNACOES)
df_qualiar = pd.read_parquet(PATH_QUALIAR)

print(f'Serie de internacoes: {df_internacoes.shape}')
print(f'  Periodo: {df_internacoes["data_dia"].min()} a {df_internacoes["data_dia"].max()}')
print(f'\nSerie de qualidade do ar: {df_qualiar.shape}')
print(f'  Periodo: {df_qualiar["data"].min()} a {df_qualiar["data"].max()}')

Serie de internacoes: (4018, 3)
  Periodo: 2008-01-01 00:00:00 a 2018-12-31 00:00:00

Serie de qualidade do ar: (2557, 12)
  Periodo: 2012-01-01 00:00:00 a 2018-12-31 00:00:00


## Preparação das datas e tratamento inicial

As colunas de data são padronizadas para o mesmo formato. As duas fontes são mantidas **separadas** nesta etapa:

- **`df_intern`** — série de internações completa (2008–2018). Todas as features endógenas (lags, janelas móveis, STL, calendário) são calculadas sobre essa série, aproveitando o histórico de 2008–2011 como contexto temporal.
- **`df_air`** — série de qualidade do ar (2012–2018). As features atmosféricas são calculadas aqui.

O **merge entre as duas fontes ocorre apenas na montagem do dataset final**, de modo que os dados de 2008–2011 contribuem como histórico para as features endógenas sem serem descartados antes do cálculo.



In [10]:
df_internacoes['data'] = pd.to_datetime(df_internacoes['data_dia']).dt.normalize()
df_qualiar['data'] = pd.to_datetime(df_qualiar['data']).dt.normalize()

# ---------------------------------------------------------------------------
# DataFrame de internacoes: cobre toda a serie historica (2008-2018).
# Features endogenas serao calculadas aqui, aproveitando 2008-2011 como
# historico para lags longos (lag_365) e janela STL (~3 anos).
# ---------------------------------------------------------------------------
df_intern = (
    df_internacoes[['data', 'num_internacoes']]
    .drop_duplicates(subset='data')
    .sort_values('data')
    .reset_index(drop=True)
)

print(f'Serie de internacoes (df_intern): {df_intern.shape}')
print(f'  Periodo: {df_intern["data"].min().date()} a {df_intern["data"].max().date()}')

# Verifica continuidade
datas_esperadas = pd.date_range(df_intern['data'].min(), df_intern['data'].max(), freq='D')
datas_faltantes = datas_esperadas.difference(df_intern['data'])
print(f'  Dias faltantes: {len(datas_faltantes)}')
if len(datas_faltantes) > 0:
    print(f'  Primeiros faltantes: {sorted(datas_faltantes)[:5]}')

# ---------------------------------------------------------------------------
# DataFrame de qualidade do ar: cobre 2012-2018.
# Features atmosfericas serao calculadas aqui.
# ---------------------------------------------------------------------------
df_air = (
    df_qualiar
    .drop_duplicates(subset='data')
    .sort_values('data')
    .reset_index(drop=True)
)

print(f'\nSerie de qualidade do ar (df_air): {df_air.shape}')
print(f'  Periodo: {df_air["data"].min().date()} a {df_air["data"].max().date()}')

# Tratamento de NaN nas variaveis atmosfericas
# As estacoes de monitoramento apresentam poucas falhas pontuais (< 1% dos dias).
# Interpolacao linear e usada para preencher essas lacunas e evitar propagacao
# de NaN nas rolling windows longas (120-150 dias).
vars_atmosfericas = ['no', 'no2', 'so2', 'pm2_5', 'nox', 'ur', 'temp', 'o3', 'co', 'pm10']
n_nulos_antes = df_air[vars_atmosfericas].isnull().sum()
df_air[vars_atmosfericas] = df_air[vars_atmosfericas].interpolate(method='linear', limit_direction='both')
n_nulos_depois = df_air[vars_atmosfericas].isnull().sum()

print(f'\nNaN em variaveis atmosfericas (antes -> depois da interpolacao):')
for var in vars_atmosfericas:
    print(f'  {var}: {n_nulos_antes[var]} -> {n_nulos_depois[var]}')

Serie de internacoes (df_intern): (4018, 2)
  Periodo: 2008-01-01 a 2018-12-31
  Dias faltantes: 0

Serie de qualidade do ar (df_air): (2557, 12)
  Periodo: 2012-01-01 a 2018-12-31

NaN em variaveis atmosfericas (antes -> depois da interpolacao):
  no: 23 -> 0
  no2: 23 -> 0
  so2: 23 -> 0
  pm2_5: 41 -> 0
  nox: 23 -> 0
  ur: 23 -> 0
  temp: 23 -> 0
  o3: 23 -> 0
  co: 23 -> 0
  pm10: 23 -> 0


## Definição da variável alvo

O alvo do modelo é o número de internações no dia corrente (`D0`). A partir daqui, todas as features são construídas para prever esse valor utilizando apenas informações observáveis até o instante da previsão.


In [11]:
df_intern['target'] = df_intern['num_internacoes'].copy()

print('Alvo (target) — estatisticas descritivas:')
print(df_intern['target'].describe())

Alvo (target) — estatisticas descritivas:
count    4018.000000
mean       42.037830
std        20.261867
min         6.000000
25%        27.000000
50%        38.000000
75%        53.000000
max       171.000000
Name: target, dtype: float64


## Features endógenas — lags da série de internações

Nesta seção são criadas variáveis de memória temporal da própria série de internações. Cada `lag_k` representa o valor observado `k` dias antes de `D0`.

Essas features ajudam o modelo a capturar persistência de curto prazo, recorrência semanal e padrões de escala mais longa.


In [12]:
for k in [1, 2, 3]:
    df_intern[f'lag_{k}'] = df_intern['num_internacoes'].shift(k)

# Memoria semanal: recorrencia semanal foi um dos sinais mais fortes na EDA
for k in [7, 14, 21, 28]:
    df_intern[f'lag_{k}'] = df_intern['num_internacoes'].shift(k)

# Memoria mensal e anual: capturam ciclos de escala mais longa
# Com a serie a partir de 2008, o lag_365 sera nao-nulo ja desde 2009,
# e para dados de 2012+ tera historico completo de 1 ano disponivel.
for k in [30, 365]:
    df_intern[f'lag_{k}'] = df_intern['num_internacoes'].shift(k)

# Resumo dos lags criados
lag_cols = [c for c in df_intern.columns if c.startswith('lag_')]
print(f'Lags criados ({len(lag_cols)}): {lag_cols}')
print(f'\nNaN por feature (linhas sem historico suficiente):')
for c in lag_cols:
    print(f'  {c}: {df_intern[c].isnull().sum()} NaNs')


Lags criados (9): ['lag_1', 'lag_2', 'lag_3', 'lag_7', 'lag_14', 'lag_21', 'lag_28', 'lag_30', 'lag_365']

NaN por feature (linhas sem historico suficiente):
  lag_1: 1 NaNs
  lag_2: 2 NaNs
  lag_3: 3 NaNs
  lag_7: 7 NaNs
  lag_14: 14 NaNs
  lag_21: 21 NaNs
  lag_28: 28 NaNs
  lag_30: 30 NaNs
  lag_365: 365 NaNs


## Features endógenas — janelas móveis da série de internações

Aqui são criadas estatísticas móveis da série de internações, sempre calculadas sobre a série deslocada em `1` dia, para impedir que o valor de `D0` entre na construção da feature.

São incluídas medidas de nível recente e de variabilidade recente da série.


In [13]:
serie_shifted = df_intern['num_internacoes'].shift(1)

# Nivel recente: resumem o patamar local da serie e ajudam a capturar
# ondas epidemiologicas de diferentes escalas
for w in [7, 14, 30]:
    df_intern[f'rolling_mean_{w}'] = serie_shifted.rolling(w).mean()

# Variabilidade recente: capturam mudancas de volatilidade, importantes
# numa serie com quebras estruturais e extremos
for w in [7, 30]:
    df_intern[f'rolling_std_{w}'] = serie_shifted.rolling(w).std()

# Resumo das rolling windows criadas
roll_cols = [c for c in df_intern.columns if c.startswith('rolling_')]
print(f'Rolling windows criadas ({len(roll_cols)}):')
for c in roll_cols:
    print(f'  {c}: {df_intern[c].isnull().sum()} NaNs')


Rolling windows criadas (5):
  rolling_mean_7: 7 NaNs
  rolling_mean_14: 14 NaNs
  rolling_mean_30: 30 NaNs
  rolling_std_7: 7 NaNs
  rolling_std_30: 30 NaNs


## Features de sinalização de pico

Quatro features que sinalizam a **entrada em períodos de alta** na série de internações. Todas derivadas de `lag_1` (valor do dia anterior), sem qualquer uso de `D0`.

| Feature | Definição | Intuição |
|---|---|---|
| `diff_7` | `lag_1 − lag_7` | Taxa de variação semanal: positivo indica aceleração |
| `rolling_max_7` | `max(lag_1, …, lag_7)` | Nível máximo recente nos últimos 7 dias |
| `accel_ratio` | `rolling_mean_7 / rolling_mean_30` | Razão curto/longo: >1 indica tendência ascendente local |
| `dias_desde_pico` | Dias desde o último `lag_1 ≥ p90(treino)` | Posição no ciclo de pico sazonal |

**Anti-leakage:** `PEAK_THRESHOLD` é calculado apenas sobre o período de treino (`≤ 2016-12-31`), nunca sobre validação ou teste.

In [14]:
# Limiar de pico: p90 calculado APENAS sobre o período de treino (≤ 2016-12-31)
# para garantir que não há leakage de informação de validação/teste.
TRAIN_END_FE = pd.Timestamp('2016-12-31')
PEAK_THRESHOLD = np.percentile(
    df_intern.loc[df_intern['data'] <= TRAIN_END_FE, 'num_internacoes'], 90
)
print(f'Limiar de pico (p90 treino, até {TRAIN_END_FE.date()}): {PEAK_THRESHOLD:.1f} internações')

# serie_shifted já foi definida acima como shift(1) — valor do dia anterior (D-1)
# Todas as features abaixo partem de serie_shifted, garantindo anti-leakage.

# ── diff_7: taxa de variação semanal ────────────────────────────────────────
# lag_1 - lag_7 = valor D-1 menos valor D-7.
# Já calculados individualmente; subtração direta entre colunas existentes.
df_intern['diff_7'] = df_intern['lag_1'] - df_intern['lag_7']

# ── rolling_max_7: nível máximo nos últimos 7 dias ─────────────────────────
# max(D-1, D-2, …, D-7). Janela de 7 dias sobre a série deslocada em 1 dia.
df_intern['rolling_max_7'] = serie_shifted.rolling(7).max()

# ── accel_ratio: razão média curto / médio prazo ───────────────────────────
# rolling_mean_7 e rolling_mean_30 já foram calculados acima (shift(1) interno).
# Evita divisão por zero com np.where.
df_intern['accel_ratio'] = np.where(
    df_intern['rolling_mean_30'] > 0,
    df_intern['rolling_mean_7'] / df_intern['rolling_mean_30'],
    1.0  # razão neutra quando denominador é zero (início da série)
)

# ── dias_desde_pico: dias desde o último pico em lag_1 ─────────────────────
# Conta quantos dias se passaram desde o último dia em que D-1 ≥ PEAK_THRESHOLD.
is_peak = (serie_shifted >= PEAK_THRESHOLD).values
dias = np.zeros(len(is_peak), dtype=float)
count = 0
for i, peak in enumerate(is_peak):
    if peak:
        count = 0
    else:
        count += 1
    dias[i] = count
df_intern['dias_desde_pico'] = dias

# Resumo
peak_cols = ['diff_7', 'rolling_max_7', 'accel_ratio', 'dias_desde_pico']
print(f'\nFeatures de pico criadas ({len(peak_cols)}):')
for c in peak_cols:
    n_nulos = df_intern[c].isnull().sum()
    print(f'  {c}: {n_nulos} NaNs | '
          f'min={df_intern[c].min():.2f}, max={df_intern[c].max():.2f}, '
          f'mean={df_intern[c].mean():.2f}')


Limiar de pico (p90 treino, até 2016-12-31): 73.0 internações

Features de pico criadas (4):
  diff_7: 7 NaNs | min=-107.00, max=84.00, mean=-0.04
  rolling_max_7: 7 NaNs | min=16.00, max=171.00, mean=56.75
  accel_ratio: 0 NaNs | min=0.66, max=1.41, mean=1.00
  dias_desde_pico: 0 NaNs | min=0.00, max=1057.00, mean=226.40


## Features de calendário e sazonalidade

Esta etapa adiciona atributos determinísticos baseados na própria data, como dia da semana, mês, semana epidemiológica, estação do ano e indicador de período sazonal crítico.

Como essas informações são conhecidas antecipadamente, elas não oferecem risco de vazamento de informação.


In [15]:
df_intern['dia_semana'] = df_intern['data'].dt.dayofweek

# Fim de semana: a EDA mostrou efeito semanal forte, com dias uteis muito
# acima de sabado/domingo
df_intern['fim_de_semana'] = (df_intern['dia_semana'] >= 5).astype(int)

# Mes do ano (1-12)
df_intern['mes'] = df_intern['data'].dt.month

# Semana epidemiologica (aproximacao via ISO week)
df_intern['semana_epidemiologica'] = df_intern['data'].dt.isocalendar().week.astype(int)

# Estacao do ano — hemisferio sul (Rio de Janeiro)
# Verao: Dez-Fev (0) | Outono: Mar-Mai (1) | Inverno: Jun-Ago (2) | Primavera: Set-Nov (3)
ESTACAO_HEMISFERIO_SUL = {
    12: 0, 1: 0, 2: 0,    # Verao
    3: 1, 4: 1, 5: 1,     # Outono
    6: 2, 7: 2, 8: 2,     # Inverno
    9: 3, 10: 3, 11: 3    # Primavera
}
df_intern['estacao'] = df_intern['mes'].map(ESTACAO_HEMISFERIO_SUL)

# ---------------------------------------------------------------------------
# Codificacao ciclica das variaveis temporais
# ---------------------------------------------------------------------------
# Mes do ano (periodo = 12, valores de 1 a 12)
df_intern['mes_sin'] = np.sin(2 * np.pi * df_intern['mes'] / 12)
df_intern['mes_cos'] = np.cos(2 * np.pi * df_intern['mes'] / 12)

# Dia da semana (periodo = 7, dayofweek: 0 = segunda-feira, 6 = domingo)
df_intern['dia_semana_sin'] = np.sin(2 * np.pi * df_intern['dia_semana'] / 7)
df_intern['dia_semana_cos'] = np.cos(2 * np.pi * df_intern['dia_semana'] / 7)

# Semana epidemiologica (periodo = 52, semanas ISO de 1 a 52/53)
df_intern['semana_epi_sin'] = np.sin(2 * np.pi * df_intern['semana_epidemiologica'] / 52)
df_intern['semana_epi_cos'] = np.cos(2 * np.pi * df_intern['semana_epidemiologica'] / 52)

# Resumo
print('Features de calendario criadas:')
for c in ['fim_de_semana', 'estacao']:
    print(f'  {c}: valores unicos = {sorted(df_intern[c].unique())}')

cyclic_cols = ['mes_sin', 'mes_cos', 'dia_semana_sin', 'dia_semana_cos',
               'semana_epi_sin', 'semana_epi_cos']
print('\nFeatures ciclicas criadas (substituem versoes inteiras no dataset final):')
for c in cyclic_cols:
    print(f'  {c}: [{df_intern[c].min():.4f}, {df_intern[c].max():.4f}]')


Features de calendario criadas:
  fim_de_semana: valores unicos = [np.int64(0), np.int64(1)]
  estacao: valores unicos = [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]

Features ciclicas criadas (substituem versoes inteiras no dataset final):
  mes_sin: [-1.0000, 1.0000]
  mes_cos: [-1.0000, 1.0000]
  dia_semana_sin: [-0.9749, 0.9749]
  dia_semana_cos: [-0.9010, 1.0000]
  semana_epi_sin: [-1.0000, 1.0000]
  semana_epi_cos: [-1.0000, 1.0000]


## Features atmosféricas finais

Nesta seção são criadas as features atmosféricas sobre **`df_air`** (qualidade do ar, 2012–2018). Em geral, são construídas como médias móveis defasadas, para representar exposição acumulada e evitar uso direto de informação futura.

A exceção documentada é `no2_lag0`, mantida conforme a definição adotada no processo anterior.



In [16]:
df_air['o3_ma_120d_shift_1d'] = df_air['o3'].shift(1).rolling(120).mean()

# Monoxido de nitrogenio (NO) — media movel de 30 dias, shift de 1 dia
df_air['no_ma_30d_shift_1d'] = df_air['no'].shift(1).rolling(30).mean()

# Monoxido de carbono (CO) — media movel de 120 dias, shift de 1 dia
df_air['co_ma_120d_shift_1d'] = df_air['co'].shift(1).rolling(120).mean()

# Dioxido de enxofre (SO2) — media movel de 150 dias, shift de 21 dias
df_air['so2_ma_150d_shift_21d'] = df_air['so2'].shift(21).rolling(150).mean()

# Oxidos de nitrogenio (NOX) — media movel de 21 dias, shift de 1 dia
df_air['nox_ma_21d_shift_1d'] = df_air['nox'].shift(1).rolling(21).mean()

# Dioxido de nitrogenio (NO2) — valor no dia D0
# Premissa: medicao de NO2 disponivel em tempo real no dia da previsao
df_air['no2_lag0'] = df_air['no2'].copy()

# Material particulado fino (PM2.5) — media movel de 30 dias, shift de 1 dia
df_air['pm2_5_ma_30d_shift_1d'] = df_air['pm2_5'].shift(1).rolling(30).mean()

# Temperatura — media movel de 14 dias, shift de 1 dia
df_air['temp_ma_14d_shift_1d'] = df_air['temp'].shift(1).rolling(14).mean()

# Umidade relativa — media movel de 60 dias, shift de 1 dia
df_air['ur_ma_60d_shift_1d'] = df_air['ur'].shift(1).rolling(60).mean()

# Resumo das features atmosfericas
atm_cols = [
    'o3_ma_120d_shift_1d', 'no_ma_30d_shift_1d', 'co_ma_120d_shift_1d',
    'so2_ma_150d_shift_21d', 'nox_ma_21d_shift_1d', 'no2_lag0',
    'pm2_5_ma_30d_shift_1d', 'temp_ma_14d_shift_1d', 'ur_ma_60d_shift_1d'
]
print('Features atmosfericas criadas (em df_air):')
for c in atm_cols:
    n_nulos = df_air[c].isnull().sum()
    print(f'  {c}: {n_nulos} NaNs ({n_nulos / len(df_air) * 100:.1f}%)')


Features atmosfericas criadas (em df_air):
  o3_ma_120d_shift_1d: 120 NaNs (4.7%)
  no_ma_30d_shift_1d: 30 NaNs (1.2%)
  co_ma_120d_shift_1d: 120 NaNs (4.7%)
  so2_ma_150d_shift_21d: 170 NaNs (6.6%)
  nox_ma_21d_shift_1d: 21 NaNs (0.8%)
  no2_lag0: 0 NaNs (0.0%)
  pm2_5_ma_30d_shift_1d: 30 NaNs (1.2%)
  temp_ma_14d_shift_1d: 14 NaNs (0.5%)
  ur_ma_60d_shift_1d: 60 NaNs (2.3%)


## Montagem do dataset final

Aqui é realizado o **merge entre `df_intern` e `df_air`** pelo inner join na coluna `data`. Isso restringe naturalmente o dataset ao período de sobreposição (2012–2018), enquanto os dados de 2008–2011 já cumpriram seu papel como histórico para o cálculo de lags, janelas móveis e STL.

Em seguida, são selecionadas as features finais e removidas as linhas iniciais sem histórico suficiente.



In [19]:
intern_feature_cols = (
    # Endogenas — lags
    [f'lag_{k}' for k in [1, 2, 3, 7, 14, 21, 28, 30, 365]]
    # Endogenas — rolling windows
    + [f'rolling_mean_{w}' for w in [7, 14, 30]]
    + [f'rolling_std_{w}' for w in [7, 30]]
    # Sinalizacao de pico
    + ['diff_7', 'rolling_max_7', 'accel_ratio', 'dias_desde_pico']
    # Calendario — codificacao ciclica
    + ['mes_sin', 'mes_cos',
       'dia_semana_sin', 'dia_semana_cos',
       'semana_epi_sin', 'semana_epi_cos',
       'fim_de_semana', 'estacao']
)

air_feature_cols = [
    'o3_ma_120d_shift_1d', 'no_ma_30d_shift_1d', 'co_ma_120d_shift_1d',
    'so2_ma_150d_shift_21d', 'nox_ma_21d_shift_1d', 'no2_lag0',
    'pm2_5_ma_30d_shift_1d', 'temp_ma_14d_shift_1d', 'ur_ma_60d_shift_1d'
]

feature_cols = intern_feature_cols + air_feature_cols

# ---------------------------------------------------------------------------
# Merge: restringe ao periodo de sobreposicao (2012-2018).
# O historico 2008-2011 ja contribuiu para o calculo das features endogenas
# em df_intern e e descartado aqui de forma controlada.
# ---------------------------------------------------------------------------
df_intern_sel = df_intern[['data', 'target'] + intern_feature_cols].copy()
df_air_sel = df_air[['data'] + air_feature_cols].copy()

df_final = pd.merge(df_intern_sel, df_air_sel, on='data', how='inner')
df_final = df_final.sort_values('data').reset_index(drop=True)

print(f'Apos merge: {df_final.shape}')
print(f'Periodo pos-merge: {df_final["data"].min().date()} a {df_final["data"].max().date()}')

# Remove linhas com NaN (historico insuficiente para alguma feature)
n_antes = len(df_final)
df_final = df_final.dropna().reset_index(drop=True)
n_depois = len(df_final)

print(f'\nLinhas antes de remover NaN: {n_antes}')
print(f'Linhas removidas (historico insuficiente): {n_antes - n_depois}')
print(f'Linhas no dataset final: {n_depois}')
print(f'Periodo final: {df_final["data"].min().date()} a {df_final["data"].max().date()}')
print(f'\nColunas ({len(df_final.columns)}):')
print(f'  Referencia temporal: data')
print(f'  Variavel alvo: target')
print(f'  Features: {len(feature_cols)}')
print(f'    - Endogenas (lags + rolling): {9 + 5}')
print(f'    - Sinalizacao de pico: 4')
print(f'    - Calendario ciclico: 6 sin/cos + 2 categoricas = 8')
print(f'    - Atmosfericas: {len(air_feature_cols)}')


Apos merge: (2557, 37)
Periodo pos-merge: 2012-01-01 a 2018-12-31

Linhas antes de remover NaN: 2557
Linhas removidas (historico insuficiente): 170
Linhas no dataset final: 2387
Periodo final: 2012-06-19 a 2018-12-31

Colunas (37):
  Referencia temporal: data
  Variavel alvo: target
  Features: 35
    - Endogenas (lags + rolling): 14
    - Sinalizacao de pico: 4
    - Calendario ciclico: 6 sin/cos + 2 categoricas = 8
    - Atmosfericas: 9


## Verificações de consistência

Antes de salvar o resultado, esta etapa faz validações importantes do dataset final, como ausência de nulos, inexistência de datas duplicadas, continuidade temporal e checagens simples de anti-leakage.


In [20]:
print('=== Verificacao de Consistencia ===\n')

# 1. Nenhum valor nulo no dataset final
n_nulos = df_final.isnull().sum().sum()
print(f'1. Valores nulos no dataset final: {n_nulos}')
assert n_nulos == 0, 'ERRO: Existem valores nulos no dataset final!'

# 2. Serie temporal sem gaps
datas_final = pd.date_range(df_final['data'].min(), df_final['data'].max(), freq='D')
gaps = datas_final.difference(df_final['data'])
print(f'2. Gaps na serie temporal: {len(gaps)}')
if len(gaps) > 0:
    print(f'   ATENCAO: {len(gaps)} dias faltantes!')

# 3. Sem datas duplicadas
n_dup = df_final['data'].duplicated().sum()
print(f'3. Datas duplicadas: {n_dup}')
assert n_dup == 0, 'ERRO: Existem datas duplicadas!'

# 4. Target com valores razoaveis (positivos)
print(f'4. Target — min: {df_final["target"].min()}, max: {df_final["target"].max()}, '
      f'media: {df_final["target"].mean():.1f}')
assert (df_final['target'] >= 0).all(), 'ERRO: Target com valores negativos!'

# 5. Anti-leakage: lag_1 deve ser altamente correlacionado com o target,
#    mas nao identico (o que indicaria uso do proprio valor de D0)
corr_lag1 = df_final['target'].corr(df_final['lag_1'])
fracao_iguais = (df_final['target'] == df_final['lag_1']).mean()
print(f'5. Anti-leakage — corr(target, lag_1): {corr_lag1:.4f} (esperado: alto, < 1)')
print(f'   Fracao target == lag_1: {fracao_iguais:.4f} (esperado: baixo)')

# 6. Rolling mean nao contaminado com D0
corr_rm7 = df_final['target'].corr(df_final['rolling_mean_7'])
print(f'6. Anti-leakage — corr(target, rolling_mean_7): {corr_rm7:.4f} (esperado: moderada)')

# 7. Resumo descritivo de todas as colunas
print(f'\n=== Resumo Descritivo ===')
display(df_final.describe().round(2))

print('\nDataset final pronto para modelagem.')

=== Verificacao de Consistencia ===

1. Valores nulos no dataset final: 0
2. Gaps na serie temporal: 0
3. Datas duplicadas: 0
4. Target — min: 6, max: 96, media: 32.5
5. Anti-leakage — corr(target, lag_1): 0.6297 (esperado: alto, < 1)
   Fracao target == lag_1: 0.0411 (esperado: baixo)
6. Anti-leakage — corr(target, rolling_mean_7): 0.7405 (esperado: moderada)

=== Resumo Descritivo ===


,data,target,lag_1,lag_2,lag_3,lag_7,lag_14,lag_21,lag_28,lag_30,...,estacao,o3_ma_120d_shift_1d,no_ma_30d_shift_1d,co_ma_120d_shift_1d,so2_ma_150d_shift_21d,nox_ma_21d_shift_1d,no2_lag0,pm2_5_ma_30d_shift_1d,temp_ma_14d_shift_1d,ur_ma_60d_shift_1d
count,2387,2387.00,2387.00,2387.00,2387.00,2387.00,2387.00,2387.00,2387.00,2387.00,...,2387.00,2387.00,2387.00,2387.00,2387.00,2387.00,2387.00,2387.00,2387.00,2387.00
mean,2015-09-25 00:00:00,32.54,32.56,32.57,32.58,32.65,32.73,32.83,32.97,33.01,...,1.56,29.77,16.05,0.34,4.36,50.51,34.44,17.60,25.96,69.74
min,2012-06-19 00:00:00,6.00,6.00,6.00,6.00,6.00,6.00,6.00,6.00,6.00,...,0.00,19.75,7.68,0.25,2.90,29.40,6.95,7.25,19.94,57.35
25%,2014-02-05 12:00:00,23.00,23.00,23.00,23.00,23.00,23.00,23.00,23.00,23.00,...,1.00,25.77,11.02,0.31,3.79,40.55,26.65,13.87,23.88,66.80
50%,2015-09-25 00:00:00,31.00,31.00,31.00,31.00,31.00,31.00,31.00,31.00,31.00,...,2.00,30.66,13.29,0.34,4.31,46.23,33.06,16.63,25.73,70.33
75%,2017-05-13 12:00:00,40.00,40.00,40.00,40.00,40.00,41.00,41.00,41.00,41.00,...,3.00,33.32,19.41,0.37,4.87,57.64,40.24,20.62,27.79,72.89
max,2018-12-31 00:00:00,96.00,96.00,96.00,96.00,96.00,96.00,96.00,96.00,96.00,...,3.00,38.28,43.44,0.46,6.41,95.57,88.47,39.14,32.29,79.00
std,NaN,13.25,13.26,13.26,13.26,13.31,13.36,13.45,13.64,13.68,...,1.12,4.72,6.89,0.04,0.74,13.30,11.24,5.39,2.62,4.24



Dataset final pronto para modelagem.


## Salvamento do dataset

Por fim, o dataframe final é salvo em formato `parquet` na camada `GoldData`, ficando pronto para uso na etapa de modelagem.


In [21]:
PATH_OUTPUT.parent.mkdir(parents=True, exist_ok=True)

# Salva em formato parquet (eficiente e preserva tipos)
df_final.to_parquet(PATH_OUTPUT, index=False)

file_size_kb = PATH_OUTPUT.stat().st_size / 1024
print(f'Dataset salvo em: {PATH_OUTPUT}')
print(f'  Shape: {df_final.shape}')
print(f'  Tamanho: {file_size_kb:.1f} KB')

Dataset salvo em: ..\..\Data\GoldData\modelDataset.parquet
  Shape: (2387, 37)
  Tamanho: 349.6 KB
